# Verifications

This notebook verifies the code I'm writing to implement the algorithms of Robertson, Sanders, Seymour and Thomas,
and verifies the reducibility of the configurations in the configuration sets of RSST and Steinberger

In [1]:
# import the configs library and other useful packages

from configs import *
from configs_demo import *
import time

# read the configuration sets
confs633=Configuration.read_configuration_file("../configurations/U_633.conf", report_success=True)
confs2822=Configuration.read_configuration_file("../configurations/U_2822.conf", report_success=True)

Read file with 633 configurations successfully
Read file with 2822 configurations successfully


## Computing the extendable edge-colorings

The RSST reducibility checking algorithm begins with computing the set of extendable edge-colorings of the ring of a configuration. The size of the set is recorded in the configuration files as part of the configuration information. Here I verify that my implementation reproduces the same results.

In [2]:
# print the first 20 recorded counts (computed by the RSST algorithm) of extendable edge-colorings in the confs2822 family
print([confs2822[j].CKcount for j in range(20)])

[16, 39, 111, 100, 98, 266, 211, 104, 251, 249, 630, 630, 665, 664, 257, 536, 538, 1430, 1582, 1600]


In [3]:
# print the first 20 counts of extendable edge-colorings in the confs2822 family, as computed by my implementations
# there are two implementations, a slow one in Python and a fast one in C
print([len(confs2822[j].extendable_edge_coloring_strings_slow()) for j in range(20)]) # Python implementation
print([len(confs2822[j].extendable_edge_coloring_strings()) for j in range(20)])      # C implementation

[16, 39, 111, 100, 98, 266, 211, 104, 251, 249, 630, 630, 665, 664, 257, 536, 538, 1430, 1582, 1600]
[16, 39, 111, 100, 98, 266, 211, 104, 251, 249, 630, 630, 665, 664, 257, 536, 538, 1430, 1582, 1600]


In [4]:
# Let's make sure my two implementations in fact compute the same set of edge-colorings
# the two implementations return the sets of colorings as lists, in different orders, so to test their equality we sort the lists
print([sorted(confs2822[j].extendable_edge_coloring_strings_slow())==sorted(confs2822[j].extendable_edge_coloring_strings()) for j in range(20)])

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]


In [5]:
# inspecting one of the sets of colorings, just for illustration
confs2822[0].extendable_edge_coloring_strings()

['grgrrr',
 'rggrrr',
 'gbbgrr',
 'grrgrr',
 'rrggrr',
 'ggggrr',
 'rgbbgr',
 'rgrrgr',
 'bbgrgr',
 'rrgrgr',
 'gggrgr',
 'brbggr',
 'bbrggr',
 'ggrggr',
 'grgggr',
 'rggggr']

In [6]:
# let's time the calculation to see how long it takes with the slow and fast implementations
counts200_slow = demo_count_extendable_colorings_slow([confs2822[j] for j in range(400)])
counts200_fast = demo_count_extendable_colorings([confs2822[j] for j in range(400)])

Counting extendable colorings (last count=4424): 100%|██████████| 400/400 [00:01<00:00, 327.04it/s] 


In [7]:
# check again that the results are identical
counts200_slow==counts200_fast

True

In [8]:
# for the fast algorithm, it takes less than a minute to go through Steinberger's entire family of 2822 configurations,
# and just a few seconds to go through the original RSST set of 633 configurations
counts2822_fast = demo_count_extendable_colorings([confs2822[j] for j in range(2822)])
counts633_fast = demo_count_extendable_colorings([confs633[j] for j in range(633)])

Counting extendable colorings (last count=3666): 100%|██████████| 633/633 [00:03<00:00, 160.15it/s] 


In [9]:
# compare the results to the recorded counts computed by the RSST algorithm
print(counts2822_fast==[confs2822[j].CKcount for j in range(2822)],
      counts633_fast==[confs633[j].CKcount for j in range(633)])

True True


## The main algorithm

The main algorithm calculates the maximal consistent subset of the set of non-extendable edge-colorings of the ring of a configuration. If this maximal consistent subset is the empty set, the configuration is D-reducible

In [10]:
# let's run the original RSST code first on the first 100 configurations in both configuration sets
# I am interfacing with the original code through the newreduce.c library, a slightly modified version of RSST's
# original reduce.c program
demo_check_reducibility_rsst([confs633[j] for j in range(100)])
demo_check_reducibility_rsst([confs2822[j] for j in range(100)])

# now run my implementation of the RSST algorithm on the same configurations
demo_check_reducibility([confs633[j] for j in range(100)])
demo_check_reducibility([confs2822[j] for j in range(100)])

Checking reducibility (RSST)... (last ring size=12): 100%|██████████| 100/100 [00:02<00:00, 47.37it/s]
Checking reducibility (RSST)... (last ring size=11): 100%|██████████| 100/100 [00:01<00:00, 90.18it/s]
Checking reducibility (DR)... (last ring size=12): 100%|██████████| 100/100 [00:10<00:00,  9.41it/s]
Checking reducibility (DR)... (last ring size=11): 100%|██████████| 100/100 [00:06<00:00, 15.87it/s]


In [11]:
confs2822[0].reducibility_check_output

{'C': [106, 10, 8, 4, 1, 0], 'M': [96, 26, 15, 11, 3, 0], 'R': 'D-reducible'}

In [13]:
# let's now inspect the output of both algorithms
print(confs633[0].reducibility_check_rsst_output)  # the output of the RSST program (adapted for interfacing with Python)
print(confs633[0].reducibility_check_output)       # the output of my implementation of the RSST algorithm

{'C': [106, 10, 8, 4, 1, 0], 'M': [95, 25, 15, 11, 3, 0], 'R': 'D-reducible'}
{'C': [106, 10, 8, 4, 1, 0], 'M': [96, 26, 15, 11, 3, 0], 'R': 'D-reducible'}


In [14]:
# are the outputs the same? no, not exactly
print("outputs are equal" if confs633[0].reducibility_check_rsst_output==confs633[0].reducibility_check_output else "outputs are not equal")

# however, the sizes of the sets C_k are the same:
print("sizes of sets C_k are equal" if confs633[0].reducibility_check_rsst_output['C']==confs633[0].reducibility_check_output['C'] else "sizes of sets C_k are not equal")

# the reason for the discrepancy in the sizes of the sets M_k have to do with the RSST algorithm treating 
# the "empty" balanced signed matching in a special way. My algorithm doesn't make that distinction and treats
# the empty balanced signed matching on an equal footing with all other balanced signed matchings. This leads to
# slightly different counts of the sets M_k. (Basically what the RSST program reports sometimes differs by 1 unit
# from the size of the set M_k they actually define in their writeup

outputs are not equal
sizes of sets C_k are equal


In [15]:
# Let's check that for all the configurations, the sizes of the sets C_k are the same for both algorithms:
for j in range(100):
    print(f"j={j}: ",end='')
    print("sizes of sets C_k are equal for both algorithms" if confs633[j].reducibility_check_rsst_output['C']==confs633[j].reducibility_check_output['C'] else "sizes of sets C_k are not equal  for both algorithms")

j=0: sizes of sets C_k are equal for both algorithms
j=1: sizes of sets C_k are equal for both algorithms
j=2: sizes of sets C_k are equal for both algorithms
j=3: sizes of sets C_k are equal for both algorithms
j=4: sizes of sets C_k are equal for both algorithms
j=5: sizes of sets C_k are equal for both algorithms
j=6: sizes of sets C_k are equal for both algorithms
j=7: sizes of sets C_k are equal for both algorithms
j=8: sizes of sets C_k are equal for both algorithms
j=9: sizes of sets C_k are equal for both algorithms
j=10: sizes of sets C_k are equal for both algorithms
j=11: sizes of sets C_k are equal for both algorithms
j=12: sizes of sets C_k are equal for both algorithms
j=13: sizes of sets C_k are equal for both algorithms
j=14: sizes of sets C_k are equal for both algorithms
j=15: sizes of sets C_k are equal for both algorithms
j=16: sizes of sets C_k are equal for both algorithms
j=17: sizes of sets C_k are equal for both algorithms
j=18: sizes of sets C_k are equal for 

In [12]:
# Now to get a feel for how fast these algorithms perform, let's run them on a few hundred configurations 
# from both configuration sets. This will take a few minutes
demo_check_reducibility_rsst([confs633[j] for j in range(300)])
demo_check_reducibility_rsst([confs2822[j] for j in range(300)])
demo_check_reducibility([confs633[j] for j in range(300)])
demo_check_reducibility([confs2822[j] for j in range(300)])

Checking reducibility (RSST)... (last ring size=14): 100%|██████████| 300/300 [00:47<00:00,  6.27it/s]
Checking reducibility (RSST)... (last ring size=11): 100%|██████████| 300/300 [00:13<00:00, 22.34it/s] 
Checking reducibility (DR)... (last ring size=14): 100%|██████████| 300/300 [04:49<00:00,  1.04it/s]
Checking reducibility (DR)... (last ring size=11): 100%|██████████| 300/300 [01:32<00:00,  3.26it/s]


## Verifying reducibility

Now that we've verified our algorithm performs similarly to the RSST algorithm, let's verify reducibility of the configurations

In [18]:
# run RSST's algorithm on their set of 633 configurations - this takes a couple of minutes
demo_check_reducibility_rsst([confs633[j] for j in range(633)])

Checking reducibility (RSST)... (last ring size=12): 100%|██████████| 633/633 [02:06<00:00,  5.02it/s]


In [22]:
# now examine the output to see how these configurations were classified by the algorithm
from collections import *   # we use the convenient Counter utility

Counter([confs633[j].reducibility_check_rsst_output['R'] for j in range(633)])

Counter({'D-reducible': 249, 'contract-reducible': 384})

In [4]:
# to duplicate the work with my own code, run my implementation of the algorithm on the RSST set of
# 633 configurations
demo_check_reducibility([confs633[j] for j in range(633)])

Checking reducibility (DR)... (last ring size=12): 100%|██████████| 633/633 [36:18<00:00,  3.44s/it]   


In [23]:
# now run RSST's algorithm on Steinberger's set of 2822 configurations - this takes about half an hour
demo_check_reducibility_rsst([confs2822[j] for j in range(2822)])

Checking reducibility (RSST)... (last ring size=12): 100%|██████████| 2822/2822 [31:17<00:00,  1.50it/s]  


In [24]:
# examine the output to see how these configurations were classified by the algorithm
Counter([confs2822[j].reducibility_check_rsst_output['R'] for j in range(2822)])

Counter({'D-reducible': 2822})

In [ ]:
# TODO: run this

# as a final check, run my version of the algorithm on Steinberger's set of 2822 configurations
# this will likely take about 12-15 hours
demo_check_reducibility([confs2822[j] for j in range(2822)])